**Casimir Effect in Quantum Field Theory with Tree Tensor Network Compression ** <br>
Joshua Tan (14506240)

---



In [ ]:
!pip install qiskit qiskit_algorithms qiskit_aer --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.4 MB/s eta 0:00:00


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA, SLSQP
from qiskit.circuit.library import efficient_su2
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
from tqdm import tqdm

**Lattice Grid Class** <br>
Takes in input of Width, Height, Lattice Spacing, Boundary Conditions <br>
Outputs grid with neighbours, plaquettes, links

---



In [ ]:
class LatticeGrid:
    def __init__(self,
                 width:int,
                 height:int,
                 spacing:float,
                 boundary_condition:str = "Casimir"):
        self.width = width
        self.height = height
        self.spacing = spacing
        self.boundary_condition = boundary_condition

        self.n_sites = self.width * self.height

        self.n_horizontal_links = self.n_sites
        self.n_vertical_links = self.width * (self.height - 1)
        self.n_links_total = self.n_horizontal_links + self.n_vertical_links

        self.n_qubits = self.n_links_total


    def site_to_coords(self, site_idx:int) -> tuple:
        if not (0 <= site_idx < self.n_sites):
            raise ValueError(f"site_idx={site_idx} out of bounds [0, {self.n_sites})")

        i = site_idx % self.width
        j = site_idx // self.width
        return (i, j)

    def coords_to_site(self, i:int, j:int) -> int:
        i = i % self.width
        if not (0 <= j < self.height):
            raise ValueError(f"j={j} out of bounds [0, {self.height})")

        return i + self.width * j

    def get_horizontal_link_index(self, i: int, j: int) -> int:
        i = i % self.width  # Handle periodic boundary
        return i + self.width * j

    def get_vertical_link_index(self, i: int, j: int) -> int:
        if j >= self.height - 1:
            raise ValueError(f"No vertical link at j={j}")

        offset = self.n_horizontal_links
        return offset + i + self.width * j

    def get_neighbours(self, i:int, j:int) -> dict:
        neighbours = {}

        neighbours["right"] = ((i + 1) % self.width, j)
        neighbours["left"] = ((i - 1) % self.width, j)

        if j < self.height - 1:
            neighbours["up"] = (i, j + 1)

        if j > 0:
            neighbours["down"] = (i, j - 1)

        return neighbours

    def get_plaquettes(self) -> list:
        plaquettes = []

        # Reminder that j is height and i is width
        for j in range(self.height - 1):
            for i in range(self.width):
                link_bottom = self.get_horizontal_link_index(i, j)
                link_right = self.get_vertical_link_index((i + 1) % self.width, j)
                link_top = self.get_horizontal_link_index(i, j + 1)
                link_left = self.get_vertical_link_index(i, j)

                plaquette = (link_bottom, link_right, link_top, link_left)
                plaquettes.append(plaquette)

        return plaquettes

    """
    Below this are visualization and debug code. This may or may not have been created with the assistace of an AI assitant who assists those who needs assistance
    """

    def visualize_ascii(self):
        """Print ASCII art visualization of lattice"""
        # Top plate markers
        print("____________" * self.width)

        # Print from top to bottom (reverse j order)
        for j in range(self.height - 1, -1, -1):
            # Print sites and horizontal links with numbers
            line = ""
            for i in range(self.width):
                line += f"({i},{j})"
                if i < self.width - 1:
                    h_link = self.get_horizontal_link_index(i, j)
                    line += f" ─{h_link:2d} ─ "
                else:
                    # Show wrap link
                    h_link = self.get_horizontal_link_index(i, j)
                    line += f" ─{h_link:2d} ─"
            print(line)

            # Print vertical links with numbers (if not bottom row)
            if j > 0:
                # Top pipe line
                pipe_line = "  │  "
                for i in range(1, self.width):
                    pipe_line += "         │  "
                print(pipe_line)

                # Link numbers line
                vline = ""
                for i in range(self.width):
                    v_link = self.get_vertical_link_index(i, j - 1)
                    vline += f" {v_link:2d}  "
                    if i < self.width - 1:
                        vline += "       "
                print(vline)

                # Bottom pipe line
                pipe_line = "  │  "
                for i in range(1, self.width):
                    pipe_line += "         │  "
                print(pipe_line)
            else:
                # Bottom plate markers
                print("▔▔▔▔▔▔▔" * self.width)


    def debug(self):
        print("Printing Statistics:")
        print("=+"*15)
        print(f"Qubit Count        | {self.n_qubits}\n" +
              f"Lattice Width      | {self.width}\n" +
              f"Lattice Height     | {self.height}\n" +
              f"Lattice Spacing    | {self.spacing}\n" +
              f"Boundary Condition | {self.boundary_condition}\n\n" +

              f"Lattice Sites         | {self.n_sites}\n" +
              f"Lattice X_Links       | {self.n_horizontal_links}\n" +
              f"Lattice Y_Links       | {self.n_vertical_links}\n" +
              f"Lattice Total Links   | {self.n_links_total}\n")

        """for site in range(self.n_sites):
            i, j = self.site_to_coords(site)
            back = self.coords_to_site(i, j)
            print(f"Site {site:2d} → ({i},{j}) → Site {back:2d}  {'✓' if back == site else '✗'}")"""

**Quantum Link Model Hamiltonian** <br>
Takes in LatticeGrid as input <br>
Constructs operators for electric and magnetic fields, total hamiltonian and return a format Qiskit could use.

---
$$ H = H_{\text{electric}} + H_{\text{magnetic}} $$ <br>
$$ H_E = \frac{g^2}{2} \sum_{i=0}^{N_{\text{links}}-1} \sigma^x_i $$ <br>
$$ H_B = \frac{1}{2g^2} \sum_{\text{plaquettes}} B_{\text{plaquettes}}^2 $$ <br>
$$ B_{\text{plaq}} = \sigma^z_1 + \sigma^z_2 - \sigma^z_3 - \sigma^z_4 $$

**Papers Used**<br>
Quantum Link Models: A Discrete Approach to Gauge Theories (Arxiv 9609042v2)

In [ ]:
class QuantumLinkModel:
    def __init__(self, lattice, coupling):
        self.lattice = lattice
        self.g_squared = coupling
        self.n_qubits = lattice.n_qubits

    def build_electric_term(self):
        coeff = self.g_squared / 2.0
        pauli_strings = []
        coefficients = []

        for qubit_idx in range(self.n_qubits):
            pauli_str = self._pauli_string_single_x(qubit_idx)

            pauli_strings.append(pauli_str)
            coefficients.append(coeff)

        H_electric = SparsePauliOp(pauli_strings, coeffs=coefficients)
        return H_electric

    def build_magnetic_term(self):
        coeff = 1.0 / (2.0 * self.g_squared)
        pauli_strings = []
        coefficients = []

        plaquettes = self.lattice.get_plaquettes()

        for plaq in plaquettes:
            link1, link2, link3, link4 = plaq

            # Term 1: 4I (identity)
            pauli_str = 'I' * self.n_qubits
            pauli_strings.append(pauli_str)
            coefficients.append(4.0 * coeff)

            # Term 2: +2σᶻ₁σᶻ₂
            pauli_str = self._pauli_string_zz(link1, link2)
            pauli_strings.append(pauli_str)
            coefficients.append(2.0 * coeff)

            # Term 3: -2σᶻ₁σᶻ₃
            pauli_str = self._pauli_string_zz(link1, link3)
            pauli_strings.append(pauli_str)
            coefficients.append(-2.0 * coeff)

            # Term 4: -2σᶻ₁σᶻ₄
            pauli_str = self._pauli_string_zz(link1, link4)
            pauli_strings.append(pauli_str)
            coefficients.append(-2.0 * coeff)

            # Term 5: -2σᶻ₂σᶻ₃
            pauli_str = self._pauli_string_zz(link2, link3)
            pauli_strings.append(pauli_str)
            coefficients.append(-2.0 * coeff)

            # Term 6: -2σᶻ₂σᶻ₄
            pauli_str = self._pauli_string_zz(link2, link4)
            pauli_strings.append(pauli_str)
            coefficients.append(-2.0 * coeff)

            # Term 7: +2σᶻ₃σᶻ₄
            pauli_str = self._pauli_string_zz(link3, link4)
            pauli_strings.append(pauli_str)
            coefficients.append(2.0 * coeff)

        # Create and return operator
        H_magnetic = SparsePauliOp(pauli_strings, coeffs=coefficients)

        return H_magnetic

    def build_hamiltonian(self):
        return self.build_electric_term() + self.build_magnetic_term()

    def _pauli_string_single_x(self, qubit_index: int) -> str:
        parts = ['I'] * self.n_qubits
        parts[qubit_index] = 'X'
        return ''.join(parts)

    def _pauli_string_zz(self, qubit1: int, qubit2: int) -> str:
        pauli = ['I'] * self.n_qubits
        pauli[qubit1] = 'Z'
        pauli[qubit2] = 'Z'
        return ''.join(pauli)

**Gauss Law Implementation**<br>
Reduces quantum Hibert Space/States drastically<br>
Additionally, enforces U(1) Gauge Symmetry

---

$$ G_{i,j} = E_{\text{right}} - E_{\text{left}} + E_{\text{up}} - E_{\text{down}} = 0 $$ <br>
$$ G_{i,j} = \sigma^x_{\text{right}} - \sigma^x_{\text{left}} + \sigma^x_{\text{up}} - \sigma^x_{\text{down}} $$ <br>

In [ ]:
class GaussLaw:
    def __init__(self, lattice):
        self.lattice = lattice
        self.n_qubits = lattice.n_qubits
        self.n_sites = lattice.n_sites

    def build_gauss_operators(self):
        gauss_ops = []

        for site_idx in range(self.n_sites):
            i, j = self.lattice.site_to_coords(site_idx)
            neighbours = self.lattice.get_neighbours(i, j)
            pauli_strings = []
            coefficients = []

            # +σ (Flux Leaving)
            if "right" in neighbours:
                i_right, j_right = neighbours["right"]
                link_idx = self.lattice.get_horizontal_link_index(i, j)
                pauli_str = self._pauli_string_single_x(link_idx)
                pauli_strings.append(pauli_str)
                coefficients.append(1.0)

            # -σ (Flux Entering)
            if "left" in neighbours:
                i_left, j_left = neighbours["left"]
                link_idx = self.lattice.get_horizontal_link_index(i_left, j_left)
                pauli_str = self._pauli_string_single_x(link_idx)
                pauli_strings.append(pauli_str)
                coefficients.append(-1.0)

            # +σ (Flux Leaving)
            if "up" in neighbours:
                i_up, j_up = neighbours["up"]
                link_idx = self.lattice.get_vertical_link_index(i, j)
                pauli_str = self._pauli_string_single_x(link_idx)
                pauli_strings.append(pauli_str)
                coefficients.append(1.0)

            # -σ (Flux Entering)
            if "down" in neighbours:
                i_down, j_down = neighbours["down"]
                link_idx = self.lattice.get_vertical_link_index(i, j_down)
                pauli_str = self._pauli_string_single_x(link_idx)
                pauli_strings.append(pauli_str)
                coefficients.append(-1.0)

            if len(pauli_strings) > 0:
                G_site = SparsePauliOp(pauli_strings, coeffs=coefficients)
                gauss_ops.append(G_site)
            else:
                gauss_ops.append(None)

        return gauss_ops

    def add_gauss_penalty(self, hamiltonian, penalty_weight=100.0):
        print(f"\nAdding Gauss law penalty (λ={penalty_weight})...")

        # Build Gauss operators
        gauss_ops = self.build_gauss_operators()

        # Start with original Hamiltonian
        H_with_penalty = hamiltonian

        # Add G² penalty for each site
        n_penalties = 0
        for site_idx, G_op in enumerate(gauss_ops):
            if G_op is None:
                continue

            # G² = G†G (measures violation of Gauss law)
            G_squared = G_op.adjoint() @ G_op

            # Add weighted penalty
            H_with_penalty = H_with_penalty + penalty_weight * G_squared
            n_penalties += 1

        print(f"  Added {n_penalties} penalty terms")
        print(f"  Hamiltonian terms: {len(hamiltonian)} → {len(H_with_penalty)}")

        return H_with_penalty

    def verify_gauss_law(self, state_vector, tolerance=1e-3, verbose=True):
        gauss_ops = self.build_gauss_operators()

        max_violation = 0.0
        violations = []

        for site_idx, G_op in enumerate(gauss_ops):
            if G_op is None:
                continue

            i, j = self.lattice.site_to_coords(site_idx)
            G_matrix = G_op.to_matrix()
            result = G_matrix @ state_vector

            norm = np.linalg.norm(result)
            max_violation = max(max_violation, norm)

            if norm > tolerance:
                violations.append((i, j, norm))

        # Print results
        if verbose:
            if len(violations) == 0:
                print("✓ State satisfies Gauss law at all sites")
                print(f"Max violation: {max_violation:.6e} (< {tolerance})")
            else:
                print(f"[!] State VIOLATES Gauss law!")
                print(f"    Tolerance: {tolerance}")
                print(f"    Max violation: {max_violation:.6e}")
                print(f"    Sites violating: {len(violations)}/{len([g for g in gauss_ops if g is not None])}")
                print(f"\nViolations:")
                for i, j, norm in violations:
                    print(f"  Site ({i},{j}): ||G|ψ⟩|| = {norm:.6e}")

        # Return True only if ALL sites satisfy Gauss law
        return len(violations) == 0

    def _pauli_string_single_x(self, qubit_index: int) -> str:
        parts = ['I'] * self.n_qubits
        parts[qubit_index] = 'X'
        return ''.join(parts)

In [ ]:
lattice = LatticeGrid(width=3, height=2, spacing=0.5, boundary_condition="Casimir")
qlm = QuantumLinkModel(lattice, coupling=1.0)
gauss = GaussLaw(lattice)

lattice.debug()
lattice.visualize_ascii()

# Build magnetic term
print("\n=== Building Magnetic Term ===")
H_B = qlm.build_magnetic_term()

print(f"\nMagnetic Hamiltonian:")
print(f"Number of terms: {len(H_B)}")
print(f"Expected: {len(lattice.get_plaquettes()) * 7}")

print(f"\nFirst few terms:")
for i in range(min(5, len(H_B))):
    print(f"  {i}: {H_B.paulis[i]} with coeff {H_B.coeffs[i]}")

# Build full Hamiltonian
print("\n=== Building Full Hamiltonian ===")
H = qlm.build_hamiltonian()

print(f"\nTotal Hamiltonian:")
print(f"Number of terms: {len(H)}")
print(f"Expected: {lattice.n_links_total} (electric) + {len(lattice.get_plaquettes()) * 7} (magnetic)")

H_phys = gauss.add_gauss_penalty(H, penalty_weight=3)

Printing Statistics:
=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+
Qubit Count        | 9
Lattice Width      | 3
Lattice Height     | 2
Lattice Spacing    | 0.5
Boundary Condition | Casimir

Lattice Sites         | 6
Lattice X_Links       | 6
Lattice Y_Links       | 3
Lattice Total Links   | 9

____________________________________
(0,1) ─ 3 ─ (1,1) ─ 4 ─ (2,1) ─ 5 ─
  │           │           │  
  6           7           8  
  │           │           │  
(0,0) ─ 0 ─ (1,0) ─ 1 ─ (2,0) ─ 2 ─
▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔

=== Building Magnetic Term ===

Magnetic Hamiltonian:
Number of terms: 21
Expected: 21

First few terms:
  0: IIIIIIIII with coeff (2+0j)
  1: ZIIIIIIZI with coeff (1+0j)
  2: ZIIZIIIII with coeff (-1+0j)
  3: ZIIIIIZII with coeff (-1+0j)
  4: IIIZIIIZI with coeff (-1+0j)

=== Building Full Hamiltonian ===

Total Hamiltonian:
Number of terms: 30
Expected: 9 (electric) + 21 (magnetic)

Adding Gauss law penalty (λ=3)...
  Added 6 penalty terms
  Hamiltonian terms: 30 → 84
